# Visualising a fitted decision tree with Graphviz

**Status:** ✅ Runs end to end

**Demonstrates:** Tree internals · Graphviz export. Reaching into `model.tree_.children_left` / `.feature` / `.threshold` is a good way to see how a fitted tree is stored.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.


In [ ]:

import numpy as np

# BUGFIX: the original used `from sklearn.externals.six import StringIO`.
# `sklearn.externals` was deprecated in 0.21 and removed in 0.23, so the cell
# raised ImportError on any current scikit-learn. StringIO has been in the
# standard library since Python 3.
from io import StringIO


## Data

A tiny categorical table: which app people prefer, broken down by gender and
age. Small enough to fit a tree that memorises it completely, which is the
point - the structure is the subject, not the accuracy.

In [ ]:

# Rows: 0 = favourite app, 1 = gender, 2 = age
# Columns: Pokemon, WhatsApp, Monday
table = np.array([
    [1, 0, 0, 0, 1, 0, 0, 1, 0, 1],     # target: likes Pokemon
    [1, 1, 0, 1, 1, 0, 1, 0, 0, 0],     # gender
    [51, 28, 31, 35, 27, 32, 46, 18, 28, 33],   # age
], dtype=float)

X = table[1:].T          # (10 samples, 2 features)
y = table[0].astype(int)

feature_names = ["gender", "age"]
print("X", X.shape, "| y", y.shape, "| classes", np.unique(y))


## Fit an entropy-based tree and inspect its internals

Reaching into `model.tree_` is the most direct way to see how a fitted tree is actually stored: one row per node, with parallel arrays for the split feature, the threshold, and the child pointers. A node whose `feature` is `-2` is a leaf.

In [ ]:

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=1, random_state=0)
model.fit(X, y)
print("training accuracy:", model.score(X, y))

tree = model.tree_
print(f"node_count={tree.node_count}  max_depth={tree.max_depth}")
print(f"{'node':>4} {'feature':>8} {'threshold':>10} {'left':>5} {'right':>6} {'impurity':>9}")
for node in range(tree.node_count):
    label = "leaf" if tree.feature[node] == -2 else feature_names[tree.feature[node]]
    print(f"{node:4d} {label:>8} {tree.threshold[node]:10.3f} "
          f"{tree.children_left[node]:5d} {tree.children_right[node]:6d} "
          f"{tree.impurity[node]:9.4f}")


## Export to Graphviz

`export_graphviz` writes DOT source, which is rendered here to PNG. `pydotplus` and `graphviz` are both required.

In [ ]:
from sklearn.tree import export_graphviz

dot_buffer = StringIO()
export_graphviz(
    model,
    out_file=dot_buffer,
    feature_names=feature_names,
    class_names=["not-pokemon", "pokemon"],
    filled=True,
    rounded=True,
    special_characters=True,
)
dot_source = dot_buffer.getvalue()
print(dot_source[:400])
print("...")

try:
    import pydotplus
    from IPython.display import Image, SVG

    graph = pydotplus.graph_from_dot_data(dot_source)
    png = graph.create_png()
    if png:
        display(Image(png))
    else:
        display(SVG(graph.create_svg().decode()))
except ImportError as exc:
    # A missing renderer is an environment problem, not a code problem.
    print(f"Graphviz rendering skipped ({exc}). The DOT source above is complete.")
    print("Install the renderer with: pip install pydotplus graphviz")
    print("plus the system graphviz package.")


## Reading the DOT source

The source is plain text and is often the most useful output of all. Each node
carries the split, the sample counts reaching it, and the class distribution;
each edge is labelled with the branch condition. The root here splits on age.